* Import necessary libraries

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTENC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

* Load data

In [ ]:
data = pd.read_csv("/content/datasets/diabetes_cleaned.csv")
print(data.info())
print(data.duplicated().sum())
print(data.gender.unique())

* Separate Feature and Target variables

In [ ]:
X = data.drop("diabetes", axis=1)
y = data["diabetes"]

In [ ]:
X.dtypes

*Two variables, 'Gender' and 'Smoking history' to be encoded*

* Split data into train and test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# check on split data

print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

* Separate feature types

In [ ]:
categorical_features = ["gender", "smoking_history"]
numerical_features = ["age", "hypertension", "heart_disease", "bmi", "HbA1c_level", "blood_glucose_level"]

## Logistic Regression Model
------
(Baseline Model)

* Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

* Logistic Regression pipeline

In [ ]:
log_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

* Training and prediction

In [ ]:
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

* Accuracy check

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

*A higher accuracy in this case does not necessarily indicate that the model performs well, due to the substantial class imbalance between diabetic and non-diabetic observations. Since the majority class consists of non-diabetic observations, the model can achieve relatively high accuracy by predicting more observations as non-diabetic, while potentially failing to correctly identify diabetic cases.*

In [ ]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

# Classification report
print(classification_report(y_test, y_pred))

*The baseline Logistic Regression model distinguishes non-diabetic cases very well, but its ability to detect diabetic cases is substantially weaker, with 63% recall.*

* Handling class imbalance

(Balanced Logistic Regression model)

In [ ]:
log_reg_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight={0: 1, 1: 3},
        max_iter=1000
    ))
])

In [ ]:
log_reg_balanced.fit(X_train, y_train)

y_pred = log_reg_balanced.predict(X_test)

In [ ]:
def report (y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)
    print(f" Accuracy: {accuracy}\n Confusion matrix: \n{cm}\n Classification report: \n{class_report}")

report(y_test, y_pred)


*class_weight="balanced", and manually tuned class weight substantially increased the model's ability to detect class 1, but it also produced many more false positives.*

## Random Forest Classifier
------
(Baseline Model)

* Preprocessing

In [ ]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

* Random Forest pipeline

In [ ]:
random_forest = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

* Train and Predict

In [ ]:
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)

In [ ]:
report(y_test, y_pred)


*The baseline Random Forest has lower diabetic recall than the class-weighted Logistic Regression, despite having higher accuracy and F1.*

(Balanced Random Forest Model)

In [ ]:
random_forest_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight={0: 1, 1: 3},
        n_estimators=100,
        random_state=42
    ))
])

In [ ]:
random_forest_balanced.fit(X_train, y_train)
y_pred = random_forest_balanced.predict(X_test)

In [ ]:
report(y_test, y_pred)

*Among the models tested so far, the baseline Random Forest achieved the highest class-1 F1-score (0.80), highest precision (0.95), and highest overall accuracy (96.91%), while maintaining a 69% recall for diabetic cases.*

* Tuned Random Forest Model

In [ ]:
param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

In [ ]:
rf_tuned = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

* Fit, inspect parameters and predicion on tuned model

In [ ]:
rf_tuned.fit(X_train, y_train)

In [ ]:
print(rf_tuned.best_params_)
print(rf_tuned.best_score_)

rf_tuned_best = rf_tuned.best_estimator_
y_pred = rf_tuned_best.predict(X_test)

In [ ]:
report(y_test, y_pred)

*Hyperparameter tuning slightly improved the Random Forest's overall accuracy and precision, but had little effect on minority-class recall. Therefore, tuning the model's structural parameters alone did not adequately address the class imbalance, indicating the need for additional imbalance-handling techniques such as SMOTENC or further decision-threshold adjustment.*

## Class Imbalance Optimisation
-----
* SMOTENC

In [ ]:
print(y_train.value_counts())
print()
print(y_train.value_counts(normalize=True))

In [ ]:
print(X.dtypes)

In [ ]:
cat_features = ["gender", "smoking_history"]
smote_nc = SMOTENC(
    categorical_features=cat_features,
    random_state=42
)

X_train_smote, y_train_smote = smote_nc.fit_resample(X_train, y_train)

*Oversampling done on training data only.*

In [ ]:
print(y_train_smote.value_counts())
print(y_train_smote.value_counts(normalize=True))

* Use balanced training data on baseline Random Forest model

In [ ]:
random_forest_smote = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

random_forest_smote.fit(X_train_smote, y_train_smote)
y_pred = random_forest_smote.predict(X_test)

In [ ]:
report(y_test, y_pred)